In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_20.pickle

In [ ]:
%%cudf.pandas.profile
### cell 20 ###

def add_gap_rows(essay):
    cols = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # pull only the needed columns on GPU
    df = train.loc[train.id == essay, cols].reset_index(drop=True)

    # compute previous end in one GPU call
    df['prev_end'] = df.discourse_end.shift(1, fill_value=-1).astype('int32')

    # build intermediate gap rows (where gap_length > 0)
    gap_mask = df.gap_length > 0
    gap_rows = df[gap_mask].assign(
        discourse_start = df.prev_end + 1,
        discourse_end   = df.discourse_start - 1,
        discourse_type  = 'Nothing',
        gap_length      = None,
        gap_end_length  = None
    )[cols]

    # build final gap row (if gap_end_length > 0)
    last = df.iloc[[-1]]
    end_mask = last.gap_end_length > 0
    end_rows = last[end_mask].assign(
        discourse_start = last.discourse_end + 1,
        discourse_end   = last.discourse_end + 1 + last.gap_end_length,
        discourse_type  = 'Nothing',
        gap_length      = None,
        gap_end_length  = None
    )[cols]

    # concatenate original + gap rows in one GPU concat
    df_essay = pd.concat([df[cols], gap_rows, end_rows], ignore_index=True)
    return df_essay.sort_values('discourse_start').reset_index(drop=True)


def print_colored_essay(essay):
    df_essay = add_gap_rows(essay)
    # bring only needed columns to CPU once
    pd_df = df_essay[['discourse_start', 'discourse_end', 'discourse_type']].to_pandas()
    # rename and convert to list of dicts in one shot
    pd_df = pd_df.rename(
        columns={'discourse_start': 'start', 'discourse_end': 'end', 'discourse_type': 'label'}
    )
    ents = pd_df.to_dict(orient='records')

    essay_path = (
        f"/home/dias-benchmarks/notebooks/erikbruin/"
        f"nlp-on-student-writing-eda/input/feedback-prize-2021/train/{essay}.txt"
    )
    with open(essay_path, 'r') as f:
        text = f.read()

    doc2 = {'text': text, 'ents': ents}
    colors = {
        'Lead': '#EE11D0', 'Position': '#AB4DE1', 'Claim': '#1EDE71',
        'Evidence': '#33FAFA', 'Counterclaim': '#4253C1',
        'Concluding Statement': 'yellow', 'Rebuttal': 'red'
    }
    options = {
        'ents': pd_df.label.unique().tolist(),
        'colors': colors
    }
    # ... (rest of visualization logic)
